# Phase 7 — OOP: the `FraudPredictor` class
### PaySim Mobile-Money Fraud Detection

**Goal:** turn the notebook model into one reusable Python class that takes **one raw transaction** and returns a **fraud decision**.
The same class will sit inside the FastAPI service in the next phase, so we write it once and test it here.

**Before you run this:** Phases 4–5 are done, so `models/` contains `xgb_fraud.json` + `model_meta.json`
(and optionally `xgb_fraud_tuned.json` + `model_meta_tuned.json`), and `data/processed/paysim_features.parquet` exists.

**Why a class, not a function?** The model, the feature list, the threshold and the destination history all
belong together and must be loaded **once**, then used for many predictions. A class holds that state and the
methods that use it.

**Structure**
- Part A — OOP in 10 minutes: class, `__init__`, methods, `@classmethod`, `@dataclass`, custom exception (Tasks 1–3)
- Part B — Write the module (Task 4)
- Part C — Load it and predict (Tasks 5–6)
- Part D — Prove it is correct: validation tests, train/serve parity, decision rule (Tasks 7–9)
- Part E — Reason codes: reuse what SHAP taught us (Task 10)
- Part F — Reflect (Task 11)

## Setup

In [1]:
import json
import logging
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

DATA_DIR = next(p for p in (Path("data/processed"), Path("../data/processed"))
                if (p / "paysim_features.parquet").exists())
MODEL_DIR = DATA_DIR.parent.parent / "models"
TARGET, SPLIT_KEY = "isFraud", "step"
print("data :", DATA_DIR.resolve())
print("model:", MODEL_DIR.resolve())

data : /Users/neeraj/Desktop/fraud_detection/data/processed
model: /Users/neeraj/Desktop/fraud_detection/models


---
# Part A — OOP in 10 minutes

| Word | Meaning | In our project |
|---|---|---|
| **class** | a blueprint | `FraudPredictor` |
| **object / instance** | one thing built from the blueprint | `predictor = FraudPredictor(...)` |
| **attribute** | data stored on the object | `self.threshold`, `self.features` |
| **method** | a function that uses that data | `predict()`, `build_features()` |
| **`__init__`** | runs when the object is created | stores the model and settings |
| **`@classmethod`** | alternative constructor | `FraudPredictor.from_artifacts(path)` |
| **`@staticmethod`** | method that needs no object data | `validate(txn)` |
| **`@dataclass`** | a class that only carries data | `Prediction` |
| **custom exception** | an error with a meaningful name | `InvalidTransactionError` |

## Task 1 — A tiny class
Read, run, then change the `risk_level` rule yourself.

In [2]:
class Threshold:
    def __init__(self, value: float) -> None:
        self.value = value

    def decide(self, probability: float) -> bool:
        return probability >= self.value

    def risk_level(self, probability: float) -> str:
        if probability >= self.value:
            return "HIGH"
        if probability >= self.value * 0.5:
            return "MEDIUM"
        return "LOW"


rule = Threshold(0.9959)
for p in (0.30, 0.80, 0.9975):
    print(f"p={p:<7} flag={rule.decide(p)!s:<5} level={rule.risk_level(p)}")

p=0.3     flag=False level=LOW
p=0.8     flag=False level=MEDIUM
p=0.9975  flag=True  level=HIGH


**Expected output:** `p=0.3 flag=False level=LOW`, `p=0.8 flag=False level=MEDIUM`, `p=0.9975 flag=True level=HIGH`.

> The object **remembers** `value`. A plain function would need the threshold passed in on every call.

## Task 2 — `@classmethod` and `@dataclass`
`from_artifacts` builds an object from files, so callers never touch `__init__` details.
`Prediction` is `frozen=True` so a result cannot be changed after it is created.

In [3]:
from dataclasses import dataclass, asdict


@dataclass(frozen=True)
class Result:
    probability: float
    is_fraud: bool


class Scorer:
    def __init__(self, threshold: float) -> None:
        self.threshold = threshold

    @classmethod
    def from_meta_file(cls, path: Path) -> "Scorer":
        return cls(json.loads(Path(path).read_text())["threshold"])

    def score(self, probability: float) -> Result:
        return Result(probability, probability >= self.threshold)


meta_path = MODEL_DIR / ("model_meta_tuned.json" if (MODEL_DIR / "model_meta_tuned.json").exists() else "model_meta.json")
scorer = Scorer.from_meta_file(meta_path)
result = scorer.score(0.9999)
print(scorer.threshold, result, asdict(result))

try:
    result.is_fraud = False
except Exception as exc:
    print("cannot modify a frozen dataclass ->", type(exc).__name__)

0.21611617505550385 Result(probability=0.9999, is_fraud=True) {'probability': 0.9999, 'is_fraud': True}
cannot modify a frozen dataclass -> FrozenInstanceError


**Expected output:** your saved threshold, then `Result(probability=0.9999, is_fraud=True)`, its dict, then `FrozenInstanceError`.

## Task 3 — Custom exceptions
In an API, "your request is wrong" (HTTP 422) and "our server broke" (HTTP 500) must be told apart.
A named exception lets the API catch **only** bad input.

In [4]:
class InvalidTransactionError(ValueError):
    pass


def check_amount(amount: float) -> None:
    if not amount >= 0:
        raise InvalidTransactionError("amount must be zero or positive")


for value in (250.0, -5.0):
    try:
        check_amount(value)
        print(value, "-> ok")
    except InvalidTransactionError as exc:
        print(value, "-> rejected:", exc)

250.0 -> ok
-5.0 -> rejected: amount must be zero or positive


**Expected output:** `250.0 -> ok` then `-5.0 -> rejected: amount must be zero or positive`.

> PaySim contains a few zero-amount transactions (check `(df.amount == 0).sum()`). The model trained on them, so we reject only impossible values (negative, NaN), not unusual ones.

---
# Part B — Write the module

## Task 4 — Create `fraud_predictor.py`
The next cell writes the module file next to this notebook. **Read every part before you run it** — the walkthrough is below.

| Piece | Job |
|---|---|
| `REQUIRED_FIELDS`, `ALLOWED_TYPES` | what a valid incoming transaction looks like |
| `InvalidTransactionError` | named error for bad input |
| `DestinationHistory` | remembers how many transactions each destination received **in each step**, so `dest_prior_txn_count` = transactions in strictly earlier steps — the exact training rule. Production: Redis / feature store |
| `Prediction` | frozen result object: probability, decision, risk band, threshold used |
| `FraudPredictor.from_artifacts` | loads model + metadata; uses the **tuned** files if they exist |
| `FraudPredictor.validate` | rejects missing fields, wrong type, negative or NaN amount or balance (zero amount is allowed: PaySim has such rows) |
| `FraudPredictor.build_features` | **the same formulas as feature engineering**, then selects columns in the saved order |
| `FraudPredictor.predict` | validate → features → probability → compare with the saved threshold |
| `FraudPredictor.predict_batch` | the same for a DataFrame |

> **Design rule:** the feature list and threshold are **read from the metadata file**, never typed into the class.
> Retrain the model, save new files, and the class follows — no code change.

In [5]:
%%writefile fraud_predictor.py
"""FraudPredictor: turns one raw PaySim transaction into a fraud decision.

Loads the artifacts saved by Phase 4/5 (model JSON + metadata JSON), rebuilds the
decision-time features exactly as in feature engineering, and applies the saved threshold.
"""
from __future__ import annotations

import json
import logging
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable, Mapping

import numpy as np
import pandas as pd
import xgboost as xgb

logger = logging.getLogger("fraud_predictor")

REQUIRED_FIELDS = ("step", "type", "amount", "oldbalanceOrg", "nameDest")
ALLOWED_TYPES = ("TRANSFER", "CASH_OUT")
NEAR_EMPTY_THRESHOLD = 0.95


class InvalidTransactionError(ValueError):
    """Raised when an incoming transaction is missing fields or has impossible values."""


class DestinationHistory:
    """Transactions received per destination account, kept per time step.

    Training defines dest_prior_txn_count as the number of transactions the destination received in
    strictly EARLIER steps. Keeping counts per step lets serving apply the identical rule.
    In production this state lives in Redis or a feature store; here it is an in-memory dict.
    """

    def __init__(self, counts: Mapping[str, Mapping[int, int]] | None = None) -> None:
        self._counts: dict[str, dict[int, int]] = {d: dict(steps) for d, steps in (counts or {}).items()}

    @classmethod
    def from_dataframe(cls, history: pd.DataFrame) -> "DestinationHistory":
        counts: dict[str, dict[int, int]] = {}
        for (dest, step), n in history.groupby(["nameDest", "step"]).size().items():
            counts.setdefault(dest, {})[int(step)] = int(n)
        return cls(counts)

    def prior_count(self, name_dest: str, step: int) -> int:
        return sum(n for s, n in self._counts.get(name_dest, {}).items() if s < step)

    def record(self, name_dest: str, step: int) -> None:
        steps = self._counts.setdefault(name_dest, {})
        steps[int(step)] = steps.get(int(step), 0) + 1

    def __len__(self) -> int:
        return len(self._counts)


@dataclass(frozen=True)
class Prediction:
    fraud_probability: float
    is_fraud: bool
    risk_band: str
    threshold: float
    feature_set: str
    model_name: str
    reasons: tuple = ()

    def to_dict(self) -> dict:
        return asdict(self)


class FraudPredictor:
    def __init__(self, model: xgb.XGBClassifier, meta: dict, history: DestinationHistory | None = None) -> None:
        self._model = model
        self._meta = meta
        self.features: list[str] = list(meta["features"])
        self.threshold: float = float(meta["threshold"])
        self.feature_set: str = meta["feature_set"]
        self.model_name: str = meta.get("model", "xgboost")
        self.history = history or DestinationHistory()

    @classmethod
    def from_artifacts(cls, model_dir: str | Path, tuned: bool = True,
                       history: DestinationHistory | None = None) -> "FraudPredictor":
        model_dir = Path(model_dir)
        suffix = "_tuned" if tuned and (model_dir / "model_meta_tuned.json").exists() else ""
        meta = json.loads((model_dir / f"model_meta{suffix}.json").read_text())
        model = xgb.XGBClassifier()
        model.load_model(str(model_dir / f"xgb_fraud{suffix}.json"))
        logger.info("loaded %s model with %d features, threshold %.4f", suffix or "baseline",
                    len(meta["features"]), meta["threshold"])
        return cls(model, meta, history)

    @staticmethod
    def validate(txn: Mapping) -> None:
        missing = [f for f in REQUIRED_FIELDS if f not in txn or txn[f] is None]
        if missing:
            raise InvalidTransactionError(f"missing fields: {missing}")
        if txn["type"] not in ALLOWED_TYPES:
            raise InvalidTransactionError(f"type must be one of {ALLOWED_TYPES}, got {txn['type']!r}")
        if not float(txn["amount"]) >= 0:
            raise InvalidTransactionError("amount must be zero or positive")
        if not float(txn["oldbalanceOrg"]) >= 0:
            raise InvalidTransactionError("oldbalanceOrg must be zero or positive")
        if int(txn["step"]) < 0:
            raise InvalidTransactionError("step cannot be negative")

    def build_features(self, raw: pd.DataFrame) -> pd.DataFrame:
        amount = raw["amount"].astype("float64")
        old_balance = raw["oldbalanceOrg"].astype("float64")
        hour = (raw["step"].astype("int64") % 24).astype("int8")
        out = pd.DataFrame(index=raw.index)
        out["log_amount"] = np.log1p(amount)
        out["is_transfer"] = (raw["type"] == "TRANSFER").astype("int8")
        out["hour"] = hour
        out["is_night"] = hour.between(0, 5).astype("int8")
        out["dest_prior_txn_count"] = pd.Series(
            [self.history.prior_count(d, int(st)) for d, st in zip(raw["nameDest"], raw["step"])],
            index=raw.index).astype("int32")
        out["dest_is_merchant"] = raw["nameDest"].str.startswith("M").astype("int8")
        out["amount_to_oldbalance_ratio"] = amount / (old_balance + 1)
        out["sender_near_empty"] = ((old_balance > 0) & (amount >= NEAR_EMPTY_THRESHOLD * old_balance)).astype("int8")
        out["sender_zero_balance"] = (old_balance == 0).astype("int8")
        return out[self.features]

    def predict_proba_batch(self, raw: pd.DataFrame) -> np.ndarray:
        for record in raw.to_dict("records"):
            self.validate(record)
        return self._model.predict_proba(self.build_features(raw))[:, 1]

    def _band(self, probability: float) -> str:
        if probability >= self.threshold:
            return "HIGH"
        if probability >= self.threshold * 0.5:
            return "MEDIUM"
        return "LOW"

    def explain_batch(self, raw: pd.DataFrame, top_k: int = 3) -> list[tuple]:
        """Top-k SHAP contributions (log-odds) per row, largest absolute effect first."""
        features = self.build_features(raw)
        contributions = self._model.get_booster().predict(xgb.DMatrix(features), pred_contribs=True)[:, :-1]
        reasons = []
        for values, contrib in zip(features.to_numpy(), contributions):
            order = np.argsort(-np.abs(contrib))[:top_k]
            reasons.append(tuple((self.features[i], float(values[i]), float(contrib[i])) for i in order))
        return reasons

    def predict(self, txn: Mapping, record_history: bool = False, explain: bool = False,
                top_k: int = 3) -> Prediction:
        self.validate(txn)
        frame = pd.DataFrame([dict(txn)])
        probability = float(self.predict_proba_batch(frame)[0])
        reasons = self.explain_batch(frame, top_k)[0] if explain else ()
        if record_history:
            self.history.record(txn["nameDest"], int(txn["step"]))
        result = Prediction(
            fraud_probability=probability,
            is_fraud=probability >= self.threshold,
            risk_band=self._band(probability),
            threshold=self.threshold,
            feature_set=self.feature_set,
            model_name=self.model_name,
            reasons=reasons,
        )
        logger.info("scored %s -> %.4f (%s)", txn.get("nameDest"), probability, result.risk_band)
        return result

    def predict_batch(self, raw: pd.DataFrame) -> pd.DataFrame:
        probabilities = self.predict_proba_batch(raw)
        return pd.DataFrame({"fraud_probability": probabilities,
                             "is_fraud": probabilities >= self.threshold}, index=raw.index)

    def __repr__(self) -> str:
        return (f"FraudPredictor(model={self.model_name!r}, feature_set={self.feature_set!r}, "
                f"n_features={len(self.features)}, threshold={self.threshold:.4f})")


def score_all(predictor: FraudPredictor, transactions: Iterable[Mapping]) -> list[Prediction]:
    return [predictor.predict(t) for t in transactions]


Overwriting fraud_predictor.py


In [6]:
sys.path.insert(0, str(Path.cwd()))
import importlib
import fraud_predictor
importlib.reload(fraud_predictor)
from fraud_predictor import DestinationHistory, FraudPredictor, InvalidTransactionError, Prediction
print("module imported from:", Path(fraud_predictor.__file__).resolve())

module imported from: /Users/neeraj/Desktop/fraud_detection/notebooks/fraud_predictor.py


**Expected output:** the full path to `fraud_predictor.py`.

---
# Part C — Load it and predict

## Task 5 — Create the predictor and the data
The features file does not keep the raw columns, so we read them from the cleaned file and **assert the two files line up row by row**.
The time split comes from the metadata, exactly as in Phases 4–5.

In [7]:
predictor = FraudPredictor.from_artifacts(MODEL_DIR)
print(predictor)
print("features:", predictor.features)

RAW_COLUMNS = ["step", "type", "amount", "oldbalanceOrg", "nameDest"]
RAW_FILE = next(DATA_DIR / n for n in ("fraud_transformed_data.parquet", "paysim_clean.parquet")
                if (DATA_DIR / n).exists())

raw = (pd.read_parquet(RAW_FILE, columns=RAW_COLUMNS + [TARGET])
         .sort_values(SPLIT_KEY, kind="stable").reset_index(drop=True))
feats = (pd.read_parquet(DATA_DIR / "paysim_features.parquet")
           .sort_values(SPLIT_KEY, kind="stable").reset_index(drop=True))

assert len(raw) == len(feats), "raw and feature files have different row counts"
assert (raw[SPLIT_KEY].to_numpy() == feats[SPLIT_KEY].to_numpy()).all(), "step order differs"
assert (raw[TARGET].to_numpy() == feats[TARGET].to_numpy()).all(), "labels differ: rows are not aligned"
assert np.allclose(np.log1p(raw["amount"].to_numpy()), feats["log_amount"].to_numpy()), "rows are not aligned"

df = pd.concat([raw, feats[predictor.features]], axis=1)

meta_file = "model_meta_tuned.json" if (MODEL_DIR / "model_meta_tuned.json").exists() else "model_meta.json"
meta = json.loads((MODEL_DIR / meta_file).read_text())
test = df[df[SPLIT_KEY] > meta["split"]["valid_end_step"]]
print(f"raw rows aligned with features: {len(df):,} | test rows: {len(test):,} | frauds: {int(test[TARGET].sum()):,}")

FraudPredictor(model='xgboost-tuned', feature_set='STANDARD', n_features=8, threshold=0.2161)
features: ['log_amount', 'is_transfer', 'hour', 'is_night', 'dest_prior_txn_count', 'amount_to_oldbalance_ratio', 'sender_near_empty', 'sender_zero_balance']


AssertionError: labels differ: rows are not aligned

**Expected output:** a one-line `FraudPredictor(model=..., feature_set=..., n_features=..., threshold=...)`,
the feature list from your metadata, and the test row/fraud counts from Phase 4.

## Task 6 — Score one real fraud and one real legit transaction
`prior_count` only counts steps **before** the transaction's own step, so scoring a stored transaction gives the same value
production would have computed at that moment.

In [ ]:
history = DestinationHistory.from_dataframe(df)
predictor.history = history
print(f"history holds {len(history):,} destination accounts")


def to_request(row: pd.Series) -> dict:
    return {col: (row[col].item() if hasattr(row[col], "item") else row[col]) for col in RAW_COLUMNS}


fraud_row = test[test[TARGET] == 1].iloc[0]
legit_row = test[test[TARGET] == 0].iloc[0]

for label, row in (("real FRAUD", fraud_row), ("real LEGIT", legit_row)):
    request = to_request(row)
    print(label, request)
    print("  ->", predictor.predict(request).to_dict(), "\n")

**Expected output:** two dicts. The fraud row should have a much higher `fraud_probability` than the legit row.
It will not always be above the threshold — one example proves nothing, which is why Task 9 tests many rows.

---
# Part D — Prove it is correct

## Task 7 — Bad input must be rejected, not scored

In [ ]:
good = to_request(legit_row)
bad_cases = {
    "missing amount":   {k: v for k, v in good.items() if k != "amount"},
    "unsupported type": {**good, "type": "PAYMENT"},
    "negative amount":  {**good, "amount": -5},
    "negative balance": {**good, "oldbalanceOrg": -1},
    "negative step":    {**good, "step": -3},
}
for name, txn in bad_cases.items():
    try:
        predictor.predict(txn)
        raise AssertionError(f"'{name}' was NOT rejected")
    except InvalidTransactionError as exc:
        print(f"{name:<18} -> rejected: {exc}")

**Expected output:** five lines, each ending in a rejection message, and no `AssertionError`.

## Task 8 — Train/serve parity: class features must equal training features
The most common production bug: the API computes features slightly differently from training.
We send 20,000 random **raw** test rows through the class and compare with the stored training features, column by column.

In [ ]:
sample = test.sample(min(20_000, len(test)), random_state=7)
served = predictor.build_features(sample[RAW_COLUMNS])
trained = sample[predictor.features]

pd.testing.assert_frame_equal(served, trained, check_dtype=False, check_exact=False, rtol=1e-6, atol=1e-9)
print(f"feature parity OK on {len(sample):,} rows, columns: {list(served.columns)}")

served_scores = predictor.predict_proba_batch(sample[RAW_COLUMNS])
trained_scores = predictor._model.predict_proba(trained)[:, 1]
assert np.allclose(served_scores, trained_scores, atol=1e-6)
print("score parity OK")

**Expected output:** `feature parity OK on 20,000 rows, columns: [...]` then `score parity OK`.

> If this fails, pandas names the column that differs. Fix the formula in `build_features`, rerun Task 4, and re-test.



## Task 9 — Does the decision rule behave on real rows?

In [ ]:
sample = test.sample(min(50_000, len(test)), random_state=42)
out = predictor.predict_batch(sample[RAW_COLUMNS])

flagged = out["is_fraud"].to_numpy()
actual = sample[TARGET].to_numpy()
tp = int((flagged & (actual == 1)).sum())
fp = int((flagged & (actual == 0)).sum())
fn = int((~flagged & (actual == 1)).sum())
print(f"rows {len(sample):,} | flagged {flagged.sum():,} | TP {tp} | FP {fp} | FN {fn}")
print(f"PR-AUC on this sample: {average_precision_score(actual, out['fraud_probability']):.4f}")

**Expected output:** one line of counts and a PR-AUC close to your Phase 5 test PR-AUC (a random sample, so not identical). Precision and recall at the saved threshold should be in the same range as Phase 5.
**Say this in an interview:** *"I proved the serving code reproduces training features exactly, including the history feature, by replaying raw rows through the class."*

---
# Part E — Reason codes with SHAP

## Task 10 — Explain every flagged transaction
In the SHAP notebook you explained *the model*. A bank must also explain *each decision* ("why was my transfer blocked?").
`predict(txn, explain=True)` returns the top SHAP contributions for that one transaction.
The class uses XGBoost's built-in `pred_contribs`, which gives the **same TreeExplainer values** without needing the `shap` package in production.

Positive value = pushed the score **towards fraud**; negative = towards legit. Units are log-odds, as in the waterfall plot.

In [ ]:
prediction = predictor.predict(to_request(fraud_row), explain=True, top_k=3)
print(f"fraud_probability = {prediction.fraud_probability:.6f} | flagged = {prediction.is_fraud}")
for feature, value, contribution in prediction.reasons:
    direction = "towards FRAUD" if contribution > 0 else "towards LEGIT"
    print(f"  {feature:<28} value = {value:>14,.4f}   shap = {contribution:+.4f}  {direction}")

try:
    import shap
    import xgboost as xgb
    row = predictor.build_features(pd.DataFrame([to_request(fraud_row)]))
    reference = shap.TreeExplainer(predictor._model)(row).values[0]
    ours = predictor._model.get_booster().predict(xgb.DMatrix(row), pred_contribs=True)[0, :-1]
    assert np.allclose(reference, ours, atol=1e-4)
    print("matches shap.TreeExplainer: True")
except ImportError:
    print("shap not installed - skipped the cross-check")

**Expected output:** the fraud probability, three feature lines with their SHAP signs, and `matches shap.TreeExplainer: True`.
The top features should agree with your SHAP bar plot (`sender_near_empty`, `amount_to_oldbalance_ratio`, `dest_prior_txn_count` are typical leaders).

> Reason codes are why interpretability is a deployment feature, not decoration: the API in the next phase returns them with every flagged transaction.

---
# Part F — Reflect (write 3–4 lines)
| Question | Your answer |
|---|---|
| Which values does the class read from the metadata file, and why not hard-code them? | |
| What does `@classmethod from_artifacts` give the caller? | |
| Why `Prediction` is frozen? | |
| Which feature needs state in production, and where would you store it? | |
| What did the parity test protect you from? | |
| How would you describe a SHAP reason code to a customer in one sentence? | |

> **Next phase:** FastAPI wraps this class: `POST /predict` → `FraudPredictor.predict()`; `InvalidTransactionError` → HTTP 422.